In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch
import matplotlib.lines as mlines


In [ ]:
df_vae_suggested = pd.read_csv('data/result_vae_send_withC.csv', usecols=['sequence'])
df_selec_suggested = pd.read_csv('data/result_r6_80_SELEC_withC.csv', usecols=['sequence'])
df_vae_suggested['sequence'] = df_vae_suggested['sequence'].apply(lambda x:x[6:-6])
df_vae_suggested['architecture'] = 'VAE'
df_selec_suggested['sequence'] = df_selec_suggested['sequence'].apply(lambda x:x[6:-6])
df_selec_suggested['architecture'] = 'SELEC'

In [ ]:
df_all_seq_1 = pd.read_csv('data/all_sequences_1.csv')
df_r6 = pd.read_csv('data/selec_r6.csv')
df_c6 = pd.read_csv('data/selec_c6.csv')
df_oxt1 = pd.read_csv('data/oxt_1.csv')
df_oxt = pd.read_csv('data/oxt.csv')


# Applying the operations as per the user's script
df_r6['architecture'] = 'exp'
df_c6['architecture'] = 'ctrl'
df_oxt1.drop(['dff','category'], axis=1, inplace=True)
df_oxt1['architecture'] = 'class 1'

df_oxt['architecture'] = 'all'
df_oxt_0 = df_oxt[df_oxt['category'] == 0]
df_oxt_0['architecture'] = 'class 0'
df_oxt_0.drop(['dff','category'], axis=1, inplace=True)

df_all_seq_1 = pd.concat([df_all_seq_1, df_oxt1, df_oxt_0, df_vae_suggested, df_selec_suggested])

# Display the final dataframe
df_all_seq_1

In [ ]:
sequences_3d_array = np.array([[[int(nuc == base) for nuc in 'ACGT'] for base in seq] for seq in df_all_seq_1['sequence']])
sequences_array = sequences_3d_array.reshape(sequences_3d_array.shape[0], -1)

# Applying PCA
pca = PCA(n_components=2)
pca_result = pca.fit_transform(sequences_array)

# Applying t-SNE
tsne = TSNE(n_components=2, random_state=0)
tsne_result = tsne.fit_transform(sequences_array)

# Creating a DataFrame for the results
result_df = pd.DataFrame({
    'PCA 1': pca_result[:, 0],
    'PCA 2': pca_result[:, 1],
    't-SNE 1': tsne_result[:, 0],
    't-SNE 2': tsne_result[:, 1],
    'architecture': df_all_seq_1['architecture']
})


custom_palette = {
    'dense': '#0000FF',  # Bright Orange
    'lstm': '#800080',   # Bright Purple
    'conv': '#FFA500',   # Contrasting Blue
    'class 1': '#008000', # Lime
    'class 0': '#FF0000', # Distinguishable Red
    'VAE':'#00FFFF',
    'SELEC':'#FF00FF',
}

# custom_palette = {
#     'dense': '#0000FF',    # Blue
#     'lstm': '#800080',     # Bright Purple
#     'conv': '#FFA500',     # Orange
#     'class 1': '#008000',  # Green
#     'class 0': '#FF0000',  # Red
#     'VAE': '#00FFFF',      # Cyan
#     'SELEC': '#FF00FF'     # Magenta
# }

# Setting font and plot sizes
plt.rcParams["font.family"] = "Arial"
plt.rcParams["font.size"] = 30  # Significantly increased font size
plt.rcParams["axes.labelsize"] = 30  # Increased label size
plt.rcParams["xtick.labelsize"] = 30  # Increased XTick size
plt.rcParams["ytick.labelsize"] = 30  # Increased YTick size

# Adjust figure size and DPI for better layout
plt.figure(figsize=(24, 12), dpi=300)  # Adjust figure size here

# PCA plot
plt.subplot(1, 2, 1)
ax1 = sns.scatterplot(
    x='PCA 1', y='PCA 2', 
    hue='architecture', 
    palette=custom_palette, 
    data=result_df,
    legend=None,  # No legend for t-SNE plot
    s=120  # Point size
)
ax1.set_xlabel('PCA 1')  # Add X label
ax1.set_ylabel('PCA 2')  # Add Y label
ax1.tick_params(axis='both', which='major', width=2)  # Making ticks thicker

# Enhancing the border of the PCA plot
for spine in ax1.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(4)  # Adjusted linewidth
    spine.set_edgecolor('black')

# t-SNE plot
plt.subplot(1, 2, 2)
ax2 = sns.scatterplot(
    x='t-SNE 1', y='t-SNE 2', 
    hue='architecture', 
    palette=custom_palette, 
    data=result_df,
    legend=None,  # No legend for t-SNE plot

    s=120  # Point size
)
ax2.set_xlabel('t-SNE 1')  # Add X label
ax2.set_ylabel('t-SNE 2')  # Add Y label
ax2.tick_params(axis='both', which='major', width=2)  # Making ticks thicker

# Enhancing the border of the t-SNE plot
for spine in ax2.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(4)  # Adjusted linewidth
    spine.set_edgecolor('black')

# Adjust layout to prevent overlapping
plt.tight_layout(pad=4.0)

# Custom legend outside of the figure
legend_elements = [
    mlines.Line2D([], [], color=custom_palette['dense'], marker='o', linestyle='None', markersize=30, label='Dense'),
    mlines.Line2D([], [], color=custom_palette['lstm'], marker='o', linestyle='None', markersize=30, label='LSTM'),
    mlines.Line2D([], [], color=custom_palette['conv'], marker='o', linestyle='None', markersize=30, label='Conv2D'),
    mlines.Line2D([], [], color=custom_palette['class 1'], marker='o', linestyle='None', markersize=30, label='Class 1'),
    mlines.Line2D([], [], color=custom_palette['class 0'], marker='o', linestyle='None', markersize=30, label='Class 0'),
    mlines.Line2D([], [], color=custom_palette['VAE'], marker='o', linestyle='None', markersize=30, label='VAE'),
    mlines.Line2D([], [], color=custom_palette['SELEC'], marker='o', linestyle='None', markersize=30, label='SELEC'),
]
plt.figlegend(handles=legend_elements, loc='lower center', ncol=7, bbox_to_anchor=(0.5, -0.0),frameon=False)

plt.show()